In [0]:
# Cargar librerias inicales
from pathlib import Path
import pandas as pd
import numpy as np

In [0]:
%sql
-- Borrar la base de datos
DROP DATABASE IF EXISTS workspace.weather_silver CASCADE; 

CREATE DATABASE IF NOT EXISTS workspace.weather_silver
COMMENT 'Capa Silver procesados'

In [0]:
# Crear la tabla
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.weather_silver.weather (
  id STRING,
  department STRING,
  location_id STRING,
  year STRING,
  month STRING,
  day STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  generationtime_ms DOUBLE,
  utc_offset_seconds BIGINT,
  timezone STRING,
  timezone_abbreviation STRING,
  elevation DOUBLE,
  -- daily_units_time STRING,
  -- daily_units_temperature_2m_max STRING,
  -- daily_units_temperature_2m_min STRING,
  -- daily_units_precipitation_sum STRING,
  -- daily_units_wind_speed_10m_max STRING,
  daily_time DATE,
  date_id INT,
  daily_temperature_2m_max DOUBLE,
  daily_temperature_2m_min DOUBLE,
  daily_precipitation_sum DOUBLE,
  daily_wind_speed_10m_max DOUBLE
)
""")

In [0]:
from pyspark.sql import functions as F

# 1. Leer Bronze
df_bronze = spark.table(
    "workspace.weather_bronze.weather"
)

# 2. Convertir daily JSON STRING -> STRUCT
df = df_bronze.withColumn(
    "daily",
    F.from_json(
        F.col("daily"),
        """
        struct<
            time:array<string>,
            temperature_2m_max:array<double>,
            temperature_2m_min:array<double>,
            precipitation_sum:array<double>,
            wind_speed_10m_max:array<double>
        >
        """
    )
)

# 3. Unir los arrays por posición
df = df.withColumn(
    "weather_daily",
    F.arrays_zip(
        F.col("daily.time"),
        F.col("daily.temperature_2m_max"),
        F.col("daily.temperature_2m_min"),
        F.col("daily.precipitation_sum"),
        F.col("daily.wind_speed_10m_max")
    )
)

# 4. Convetimos el arreglo a una fila.
#  Ejemplo ahora tenemos 1 fila con un arreglo
# # 1  | Lima       | [día1, día2, día3]
# # y lo convertiremos a 3 filas
# # 1  | Lima       | día1
# # 1  | Lima       | día2
# # 1  | Lima       | día3
df = df.withColumn(
    "weather_daily",
    F.explode(F.col("weather_daily"))
)

# 5. Seleccionar y extraer los datos
df_silver = df.select(
    "id",
    "department",
    "latitude",
    "longitude",
    "generationtime_ms",
    "utc_offset_seconds",
    "timezone",
    "timezone_abbreviation",
    "elevation",

    F.col("weather_daily.time")
        .alias("daily_time"),

    F.col("weather_daily.temperature_2m_max")
        .alias("daily_temperature_2m_max"),

    F.col("weather_daily.temperature_2m_min")
        .alias("daily_temperature_2m_min"),

    F.col("weather_daily.precipitation_sum")
        .alias("daily_precipitation_sum"),

    F.col("weather_daily.wind_speed_10m_max")
        .alias("daily_wind_speed_10m_max")
)

# 6. location_id
df_silver = df_silver.withColumn(
    "location_id",
    F.regexp_replace(
        F.lower(F.col("department")),
        " ",
        "_"
    )
)

# 7. Convertir fecha
df_silver = df_silver.withColumn(
    "daily_time",
    F.to_date("daily_time")
)

# 8. date_id
df_silver = df_silver.withColumn(
    "date_id",
    F.date_format(
        "daily_time",
        "yyyyMMdd"
    ).cast("int")
)

# 9. Año, mes y día
df_silver = (
    df_silver
    .withColumn("year", F.year("daily_time").cast("string"))
    .withColumn("month", F.month("daily_time").cast("string"))
    .withColumn("day", F.dayofmonth("daily_time").cast("string"))
)

In [0]:
df_silver.select(
    "department",
    "daily_time",
    "daily_temperature_2m_max",
    "daily_temperature_2m_min",
    "daily_precipitation_sum",
    "daily_wind_speed_10m_max"
).show(20, False)

In [0]:

# 10. Crear el dataframe 
df_silver.printSchema()

In [0]:

# 12. Guardar en la capa plata
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.weather_silver.weather"
    )

In [0]:
%sql 
select * from workspace.weather_silver.weather limit 10 